In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.s4lake_silver"
GOLD = "workspace.s4lake_gold"

def salvar(df, schema, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{schema}.{tabela}")

fato_titulos = spark.table(f"{GOLD}.fato_titulos")

base = fato_titulos.agg(
    F.max("data_corte").alias("data_corte"),

    F.sum(F.when(F.col("status") == "aberto", F.col("valor"))).alias("carteira_aberta"),

    F.sum(F.when(F.col("situacao") == "vencido", F.col("valor"))).alias("carteira_vencida"),

    F.sum(F.when(F.col("status") == "pago", F.lit(1))).alias("qtd_pagos"),

    F.sum(F.when(F.col("situacao") == "pago com atraso", 
    F.lit(1))).alias("qtd_pagos_com_atraso"),

    F.sum(F.when(F.col("status") == "pago", F.greatest(F.col("dias_atraso"), F.lit(0)))).alias("soma_atraso"),

    F.sum(F.when(F.col("status") == "pago", F.col("valor"))).alias("soma_valor_pagos"),

    F.sum(F.when(F.col("status") == "pago", F.col("valor") * F.greatest(F.col("dias_atraso"), F.lit(0)))).alias("soma_valor_x_atraso"),
)

kpis_gerais = (
    base
    .withColumn("pct_vencido", F.round(F.col("carteira_vencida") / F.col("carteira_aberta") * 100, 2))

    .withColumn("pct_pagos_com_atraso", F.round(F.col("qtd_pagos_com_atraso") / F.col("qtd_pagos") * 100, 2))

    .withColumn("atraso_medio_simples", F.round(F.col("soma_atraso") / F.col("qtd_pagos"), 2))

    .withColumn("atraso_medio_ponderado", F.round(F.col("soma_valor_x_atraso") / F.col("soma_valor_pagos"), 2))
)

salvar(kpis_gerais, GOLD, "kpis_gerais")

In [0]:
spark.sql(f"""
    SELECT pct_vencido, pct_pagos_com_atraso, atraso_medio_simples, atraso_medio_ponderado
    FROM {GOLD}.kpis_gerais
""").show()